<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


**Validation status:** built and executed against a local mock matching the warehouse's confirmed schema. Run in Colab with `HF_TOKEN` for real numbers before these exports feed the paper.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Archetype → action mapping.** A raw `decline_probability` isn't itself a decision — it becomes one by crossing it with the one caveat this whole project keeps surfacing: volume. The same model score means something different at 10,000 impressions than at 2.

| Archetype | Condition | Action | Reason code |
|---|---|---|---|
| **Refresh Priority** | decline risk ≥ 0.5, impressions ≥ 50 | Add to this week's refresh review | `model_predicted_decline_risk_confirmed_volume` |
| **Watch List** | decline risk ≥ 0.5, impressions < 50 | Monitor — don't act yet | `model_predicted_decline_risk_low_volume` |
| **Protect** | decline risk < 0.5, impressions ≥ 50 | No action, light monitoring | `stable_or_growing_real_traffic` |
| **Low Priority** | decline risk < 0.5, impressions < 50 | No action | `low_risk_low_volume` |

**The decay/refresh insight, stated plainly:** across Weeks 3-6, three independent checks kept pointing the same direction — CTR falls in a clean, monotonic curve as position worsens (a real, confirmed signal); staleness alone is a real but weak signal (mixed, non-monotonic); and any decline signal built from very low-volume content is unreliable regardless of which model computes it. The playbook below is built to act on the first, use the second as a coarse gate rather than a driver, and route the third into "watch" instead of "act."

In [1]:
%pip install -q duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

from google.colab import userdata
HF_TOKEN = userdata.get('flyrank-huggingface')
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

content_df = con.sql(f"SELECT content_hash_id, word_count, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
clients_df = con.sql(f"""
    SELECT client_hash_id, is_active, has_gsc_access, gsc_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
clients_df['gsc_data_start'] = pd.to_datetime(clients_df['gsc_data_start'])

CUTOFF = pd.Timestamp("2026-01-01")  # confirm against capstone_model.ipynb's Section 1 diagnostic if not already done
usable_clients = clients_df[
    (clients_df['is_active']) & (clients_df['has_gsc_access']) &
    (clients_df['gsc_data_start'].notna()) & (clients_df['gsc_data_start'] <= CUTOFF)
]['client_hash_id']
print(f"usable clients: {len(usable_clients)} / {len(clients_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

usable clients: 29 / 104


In [2]:
def month_features(month_str):
    fact_path = f"{REL}/fact_content_daily_performance/month={month_str}/*.parquet"
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position,
               SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks
        FROM read_parquet('{fact_path}') GROUP BY client_hash_id, content_hash_id
    """).df()

def build_pair(feat_month, label_month, cutoff_date):
    feat = month_features(feat_month)
    label = month_features(label_month)[['client_hash_id', 'content_hash_id', 'clicks']].rename(columns={'clicks': 'clicks_next'})
    d = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
    d = d.merge(content_df, on='content_hash_id', how='left')
    d = d[d['client_hash_id'].isin(usable_clients)].copy()
    d['ctr'] = np.where(d['impressions'] > 0, d['clicks'] / d['impressions'], np.nan)
    d['content_created_date'] = pd.to_datetime(d['content_created_date'])
    d['content_age_days'] = (pd.Timestamp(cutoff_date) - d['content_created_date']).dt.days
    d = d[d['content_age_days'] >= 0].copy()
    d['declined_next'] = (d['clicks_next'] < 0.85 * d['clicks']).astype(int)
    return d

train = build_pair("2026-01", "2026-02", "2026-01-31")
test  = build_pair("2026-02", "2026-03", "2026-02-28")

FEATURES = ['avg_position', 'impressions', 'ctr', 'word_count', 'content_age_days']

from sklearn.ensemble import GradientBoostingClassifier
Xtr, ytr = train[FEATURES].fillna(0), train['declined_next']
Xte, yte = test[FEATURES].fillna(0), test['declined_next']
gbc = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
test['decline_probability'] = gbc.predict_proba(Xte)[:, 1]
print(f"scored {len(test):,} content items on the held-out test slice (features=Feb, label=March)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

scored 219,714 content items on the held-out test slice (features=Feb, label=March)


In [3]:
RISK_THRESHOLD = 0.5
MIN_IMPRESSIONS = 50

def assign_archetype(row):
    high_risk = row['decline_probability'] >= RISK_THRESHOLD
    enough_volume = row['impressions'] >= MIN_IMPRESSIONS
    if high_risk and enough_volume:
        return 'Refresh Priority', 'model_predicted_decline_risk_confirmed_volume', 'add_to_refresh_review_queue', \
               "Model flags rising decline risk and this page gets enough traffic to trust the signal - add to this week's refresh review."
    if high_risk and not enough_volume:
        return 'Watch List', 'model_predicted_decline_risk_low_volume', 'monitor_do_not_act_yet', \
               "Model flags decline risk, but traffic is too low to trust the number yet - watch, don't act."
    if not high_risk and enough_volume:
        return 'Protect', 'stable_or_growing_real_traffic', 'no_action_light_monitor', \
               "Real traffic, model doesn't see decline risk - leave it alone, light monitoring only."
    return 'Low Priority', 'low_risk_low_volume', 'no_action', \
           "Low risk and low traffic - not worth attention this cycle."

archetype_results = test.apply(lambda r: pd.Series(assign_archetype(r),
    index=['archetype', 'reason_code', 'action', 'human_readable_reason']), axis=1)
test_playbook = pd.concat([test, archetype_results], axis=1)

print("archetype breakdown:")
print(test_playbook['archetype'].value_counts())
print()

queue_cols = ['client_hash_id', 'content_hash_id', 'archetype', 'decline_probability', 'impressions',
              'avg_position', 'reason_code', 'action', 'human_readable_reason']
ranked_queue = test_playbook.sort_values('decline_probability', ascending=False)[queue_cols].reset_index(drop=True)

print("top 10, Refresh Priority only:")
print(ranked_queue[ranked_queue['archetype'] == 'Refresh Priority'].head(10)[
    ['content_hash_id', 'decline_probability', 'impressions', 'human_readable_reason']
].to_string(index=False))

archetype breakdown:
archetype
Low Priority        140572
Protect              66018
Refresh Priority     10858
Watch List            2266
Name: count, dtype: int64

top 10, Refresh Priority only:
         content_hash_id  decline_probability  impressions                                                                                                      human_readable_reason
content_f054be76d023d44b             0.896902         56.0 Model flags rising decline risk and this page gets enough traffic to trust the signal - add to this week's refresh review.
content_636b54119b0ff499             0.895564         50.0 Model flags rising decline risk and this page gets enough traffic to trust the signal - add to this week's refresh review.
content_41f2230ab6d362e5             0.894163         99.0 Model flags rising decline risk and this page gets enough traffic to trust the signal - add to this week's refresh review.
content_aa5a473b0191918e             0.888615         58.0 Model flags risi

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a content editor or strategist working through a fixed weekly refresh capacity across FlyRank's existing, currently-active clients.

**For what:** ordering an already-existing weekly task (which pages to look at first), not deciding *whether* to have a refresh process at all, and not replacing an editor's judgment about any individual page.

**Where it stops being valid:**
- **Time window.** Validated on one train pair (Jan→Feb) and one held-out test pair (Feb→Mar). It has not been shown to hold across a full season, a holiday period, or a known algorithm-update month.
- **Client base.** Only covers `is_active` clients with confirmed GSC access and history predating the training window — not new clients, not clients missing `gsc_data_start`.
- **Volume floor.** Below ~50 impressions, "decline" is close to a coin-flip on small integers (Weeks 5-6's own finding) — the Watch List archetype exists specifically to keep this stuff out of the actioned queue.
- **Not causal.** Nothing here proves a specific refresh *caused* a specific outcome, or says anything about Google's ranking algorithm itself.
- **One label definition.** "Decline" means a 15%+ click-rate drop into the following month. A different threshold, or a different outcome entirely (engagement, conversions), would rank pages differently.

In [4]:
print(f"Validated window: features=2026-01/02, label=2026-02/03 (one train pair, one held-out test pair)")
print(f"Usable clients this run: {len(usable_clients)} / {len(clients_df)}")
print(f"Volume floor: {MIN_IMPRESSIONS} impressions | Risk threshold: {RISK_THRESHOLD}")
print(f"Refresh Priority queue size: {(test_playbook['archetype']=='Refresh Priority').sum()} / {len(test_playbook)} scored items")

Validated window: features=2026-01/02, label=2026-02/03 (one train pair, one held-out test pair)
Usable clients this run: 29 / 104
Volume floor: 50 impressions | Risk threshold: 0.5
Refresh Priority queue size: 10858 / 219714 scored items


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any Refresh Priority row, a human checks:**
1. Is this client still active and reachable? (`usable_clients` already filters `is_active`/`has_gsc_access`, but that's a data-freshness snapshot, not a live confirmation.)
2. Does one client dominate the top of the queue? (Week 4's baseline pass found 8 of 10 top picks from a single high-traffic client — check for this every run, not just once.)
3. Does the page's actual content still make business sense to refresh (discontinued product, deprecated service, out-of-date claims)? The model has no idea what the page is *about*.

**What must never be automated:**
- **Publishing changes without human review.** This system ranks candidates; it does not write, approve, or ship content changes.
- **Treating "Refresh Priority" as a guarantee.** It's a lead, not a verdict — Section 2's limits apply to every single row.
- **Acting on Watch List items.** By construction, these don't have enough volume to trust — that's the entire point of the archetype.
- **Any claim to a client that this predicts, causes, or proves a ranking outcome.** Directional, decision-support language only, per the public rule.

In [5]:
# concrete human-review flag: surface client concentration in the actioned queue, every run, not just once
refresh_priority = ranked_queue[ranked_queue['archetype'] == 'Refresh Priority']
top20_client_counts = refresh_priority.head(20)['client_hash_id'].value_counts()
concentrated = top20_client_counts[top20_client_counts >= 4]

if len(concentrated):
    print("CLIENT CONCENTRATION FLAG - review before acting:")
    print(concentrated)
else:
    print("No single client dominates the top 20 Refresh Priority rows this run.")

CLIENT CONCENTRATION FLAG - review before acting:
client_hash_id
client_9958f0a7ae1df715    7
client_73cda7b4e4f265ea    6
client_08a6a72ff48e62c0    5
Name: count, dtype: int64


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision@K drop.** Re-run `w05_model.ipynb`'s evaluation on a fresh month pair; if precision@10 falls meaningfully below what's on record, the model stopped generalizing.
- **Feature drift.** If the feature distributions between the training window and the current month shift substantially (checked below), the model is scoring content that doesn't look like what it learned from.
- **Base rate shift.** If the share of content actually declining month-over-month moves a lot, the whole notion of "decline" this model learned may not match current reality (a seasonal spike, an algorithm update, a client mix change).
- **Client base changes.** New clients onboarding, or `is_active`/`has_gsc_access` flipping for existing ones, changes who this is even valid for — re-check `usable_clients` every run, don't cache it.
- **Time since last retrain.** Only one train→test month pair has ever been validated — retrain and re-validate at least monthly, not on a "seems fine" basis.

In [6]:
# concrete drift check: compare feature distributions between the training window (Jan) and the
# most recent scored window (Feb) - a real, computable version of "did the input data shift?"
drift_rows = []
for f in FEATURES:
    train_mean = train[f].mean()
    test_mean = test[f].mean()
    pct_change = (test_mean - train_mean) / train_mean * 100 if train_mean != 0 else float('nan')
    drift_rows.append({'feature': f, 'train_mean': round(train_mean, 3), 'test_mean': round(test_mean, 3), 'pct_change': round(pct_change, 1)})

drift_table = pd.DataFrame(drift_rows)
print(drift_table.to_string(index=False))
print()
DRIFT_ALERT_PCT = 25
flagged = drift_table[drift_table['pct_change'].abs() >= DRIFT_ALERT_PCT]
if len(flagged):
    print(f"DRIFT ALERT (>= {DRIFT_ALERT_PCT}% change) - consider retraining before trusting this run's queue:")
    print(flagged.to_string(index=False))
else:
    print(f"No feature moved more than {DRIFT_ALERT_PCT}% between train and test windows this run.")

         feature  train_mean  test_mean  pct_change
    avg_position      15.151     13.628       -10.1
     impressions     702.177    764.922         8.9
             ctr       0.006      0.005       -13.0
      word_count    2141.487   2254.353         5.3
content_age_days     201.850    216.466         7.2

No feature moved more than 25% between train and test windows this run.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on these files.*

In [7]:
import os
import json

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# 1. the ranked queue (gitignored by design - regenerated every run)
queue_path = '../outputs/action_playbook_queue.csv'
ranked_queue.to_csv(queue_path, index=False)
print(f"wrote {len(ranked_queue):,} rows to {queue_path}")

# 2. metrics JSON (committed - this is the paper's receipt for every number in the playbook)
metrics = {
    "validated_window": {"train": "2026-01 -> 2026-02", "test": "2026-02 -> 2026-03"},
    "usable_clients": int(len(usable_clients)),
    "total_clients": int(len(clients_df)),
    "risk_threshold": RISK_THRESHOLD,
    "min_impressions_floor": MIN_IMPRESSIONS,
    "archetype_counts": test_playbook['archetype'].value_counts().to_dict(),
    "drift_check": drift_table.to_dict(orient='records'),
}
metrics_path = '../outputs/action_playbook_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"wrote {metrics_path}")

wrote 219,714 rows to ../outputs/action_playbook_queue.csv
wrote ../outputs/action_playbook_metrics.json


In [8]:
import matplotlib.pyplot as plt

TEAL = '#1B8A6B'
AMBER = '#C4622D'
INK = '#14213D'
plt.rcParams.update({'font.size': 11, 'axes.edgecolor': '#DCE2E0', 'axes.linewidth': 0.8})

# figure 1: archetype distribution
fig, ax = plt.subplots(figsize=(7, 3.8))
counts = test_playbook['archetype'].value_counts().reindex(['Refresh Priority', 'Watch List', 'Protect', 'Low Priority']).fillna(0)
colors = [AMBER, '#B8860B', TEAL, '#8A96A3']
ax.bar(counts.index, counts.values, color=colors)
ax.set_ylabel('content items')
ax.set_title('Action playbook: archetype distribution', color=INK, fontsize=12, loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
fig1_path = '../figures/archetype_distribution.png'
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"wrote {fig1_path}")

# figure 2: precision@K, model vs baseline (reused visual for the paper's Results section)
visible = (test['avg_position'] > 0).astype(int)
stale = (test['content_age_days'] >= 200).astype(int)
pos_band = pd.qcut(test.loc[visible == 1, 'avg_position'], 4, duplicates='drop')
band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')
ctr_under = pd.Series(0, index=test.index)
ctr_under.loc[visible == 1] = (test.loc[visible == 1, 'ctr'] < band_median_ctr).astype(int)
baseline_score = stale * visible * ctr_under * test['impressions']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

Ks = [10, 25, 50, 100]
baseline_p = [precision_at_k(baseline_score, yte, k) for k in Ks]
model_p = [precision_at_k(test['decline_probability'], yte, k) for k in Ks]

fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(Ks))
width = 0.35
ax.bar(x - width/2, baseline_p, width, label='Baseline (rule)', color='#8A96A3')
ax.bar(x + width/2, model_p, width, label='Model (gradient boosting)', color=TEAL)
ax.axhline(yte.mean(), color=AMBER, linestyle='--', linewidth=1, label='Base rate')
ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in Ks])
ax.set_ylabel('Precision@K')
ax.set_title('Model vs. baseline, held-out test slice', color=INK, fontsize=12, loc='left')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
fig2_path = '../figures/precision_at_k_comparison.png'
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"wrote {fig2_path}")

wrote ../figures/archetype_distribution.png


/tmp/ipykernel_716/3624308255.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  band_median_ctr = test.loc[visible == 1].groupby(pos_band)['ctr'].transform('median')


wrote ../figures/precision_at_k_comparison.png


## Self-check

- [x] Ranked actions with reason codes, in plain-English "human trusts" language
- [x] Archetype → action mapping (Refresh Priority / Watch List / Protect / Low Priority)
- [x] Intended use and limits stated explicitly
- [x] Human review checklist and an explicit no-go list
- [x] Monitoring/retrain triggers, with a real computable drift check
- [x] Queue, metrics JSON, and two figures exported to `work/outputs/` and `work/figures/`
- [ ] Run this in Colab against the real warehouse — every number above needs a real run before it's a claim
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.